# 🦀 Day 3: Memory optimization
---

Today we optimize **memory** — reducing allocations and choosing the right types.

- **Why allocations matter**: Heap vs stack, allocator overhead, cache locality
- **std::mem::size_of**: Measure type sizes
- **Reducing String allocations**: `&str`, `Cow<str>`, `Arc<str>`
- **SmallVec** and **ArrayVec**: Stack-allocated for small collections
- **Box<[T]>** vs **Vec<T>**: When you know the final size
- **Reusing allocations**: `Vec::clear()` + reuse, `with_capacity`

In [ ]:
:dep smallvec = "1.13"
:dep tinyvec = { version = "1", features = ["alloc"] }

In [ ]:
// Measuring type sizes with std::mem::size_of
use std::mem::size_of;

println!("size_of::<i32>() = {}", size_of::<i32>());
println!("size_of::<Vec<i32>>() = {}", size_of::<Vec<i32>>());
println!("size_of::<String>() = {}", size_of::<String>());
println!("size_of::<&str>() = {}", size_of::<&str>());
println!("size_of::<Option<String>>() = {}", size_of::<Option<String>>());

In [ ]:
// Reducing String allocations: prefer &str, Cow, Arc<str>
use std::borrow::Cow;

fn process<'a>(s: Cow<'a, str>) -> String {
    if s.contains("error") {
        s.to_uppercase()  // allocates only when needed
    } else {
        s.into_owned()   // or keep as-is
    }
}

let static_str = "hello";
let owned = String::from("hello error");

println!("{}", process(Cow::Borrowed(static_str)));
println!("{}", process(Cow::Owned(owned)));

In [ ]:
// SmallVec: stack for small sizes, heap for large
use smallvec::SmallVec;

// SmallVec<[T; 4]> — up to 4 elements on stack, then heap
let mut v: SmallVec<[i32; 4]> = SmallVec::new();
v.push(1);
v.push(2);
v.push(3);
println!("SmallVec (3 elements): {:?}", v);

v.push(4);
v.push(5);  // spills to heap
println!("SmallVec (5 elements): {:?}", v);

In [ ]:
// ArrayVec: fixed capacity, stack-only, no heap
use tinyvec::ArrayVec;

let mut v: ArrayVec<[i32; 8]> = ArrayVec::default();
for i in 0..8 {
    v.push(i);
}
// v.push(9);  // would panic — capacity exceeded
println!("ArrayVec: {:?}", v);

In [ ]:
// Reusing allocations: Vec::clear + reuse, with_capacity
let mut buf = Vec::with_capacity(1000);

for round in 0..5 {
    buf.clear();
    for i in 0..100 {
        buf.push(round * 100 + i);
    }
    println!("Round {}: len={}, cap={}", round, buf.len(), buf.capacity());
}
// Capacity stays 1000 — no reallocations

## 📝 Exercise: Optimize a function to minimize allocations

Given a function that builds a `Vec<String>` from many `&str` slices, optimize it to:
1. Use `Vec::with_capacity` if you know the count
2. Consider `SmallVec` if the typical size is small (< 8)
3. Use `&str` or `Cow<str>` where ownership isn't needed

In [ ]:
// YOUR CODE HERE
// fn build_strings(slices: &[&str]) -> Vec<String> {
//     slices.iter().map(|s| s.to_string()).collect()
// }
// Optimize: with_capacity, or return Vec<&str> / SmallVec<[String; 4]>

## 🎯 Key Takeaways

1. **std::mem::size_of** reveals type sizes; prefer smaller types when possible
2. **&str** and **Cow<str>** avoid unnecessary String allocations
3. **SmallVec** keeps small collections on the stack; **ArrayVec** is fixed, stack-only
4. **Vec::with_capacity** and **String::with_capacity** avoid reallocations
5. **Vec::clear()** reuses capacity; pool allocations in hot loops
6. **Box<[T]>** is ideal when the size is fixed and known

---
**Tomorrow:** SIMD